# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset includes ordered logistic regression results investigating predictors for the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the following URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the Croissant-defined dataset and inspect its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and display the dataset metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}\n")
print(f"License: {meta.license}\n")
print(f"Version: {meta.version}\n")

## 2. Data Overview

Identify the available **record sets** (tables), their `@id`s, and the fields and columns in each one. All references will use the `@id` as required by the Croissant schema.

In [ ]:
# Enumerate all available record sets, fields, and columns by their @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in this dataset. Please check the source Croissant schema for the correct structure.")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):  # single field as dict
            fields = [fields]
        if fields:
            print("  Fields:")
            for fld in fields:
                print(f"    - {fld['@id']}")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("  Columns:")
            for col in columns:
                print(f"    - {col['@id']}")
        print("")

# For this dataset, also print the number of distributions (raw files)
print(f"\nNumber of distributions found: {len(meta.distribution) if hasattr(meta, 'distribution') else 0}")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame, referencing each record set and field by its `@id`. If there are no record sets defined in the schema, attempt to list records from available distributions (raw files as fallback).

In [ ]:
# If there are no record sets, try to access the distributions as the primary data tables
from collections.abc import Iterable

dataframes = dict()

if record_sets:
    # Extract each record set by its @id
    record_set_ids = [rs['@id'] for rs in record_sets]
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame loaded for: {record_set_id}, shape: {dataframes[record_set_id].shape}")
else:
    # Fallback: Try to load main data file from the first distribution
    if hasattr(meta, 'distribution') and len(meta.distribution) > 0:
        for d in meta.distribution:
            dist_id = d['@id']
            try:
                records = list(dataset.records(source=dist_id))
                if records:
                    dataframes[dist_id] = pd.DataFrame(records)
                    print(f"Loaded records from distribution {dist_id}: shape = {dataframes[dist_id].shape}")
            except Exception as e:
                print(f"Could not load records from distribution {dist_id}: {e}")
    else:
        print("No record sets or distributions could be loaded.")

# Display columns in the first loaded DataFrame (if any)
if dataframes:
    main_id = list(dataframes.keys())[0]
    print(f"\nColumns in main data ({main_id}):")
    print(dataframes[main_id].columns.tolist())
    display = dataframes[main_id].head()
    display
else:
    print("No tabular data available to display.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data exploration — filtering rows based on a criterion, normalizing a numeric column, and grouping by a categorical field.

> Note: If there are no record sets, the main DataFrame comes from a distribution file, referenced by its `@id`.

In [ ]:
# Pick the main DataFrame for analysis
if dataframes:
    main_id = list(dataframes.keys())[0]
    df = dataframes[main_id].copy()
    print(f"Analyzing data from: {main_id}")
    # Try to determine a numeric field by checking column types
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_fields:
        # Try to convert columns to numeric where possible
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered rows with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Pick a group field (first object or categorical column)
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by {group_field_id} (first 5 groups):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(grouped_df.head())
    else:
        print("No numeric fields detected in the data. EDA cannot proceed.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field, as well as boxplots grouped by a categorical field if suitable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals():
        # Histogram of the selected numeric field
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
        plt.title(f"Distribution of '{numeric_field_id}' (filtered)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        
        # Boxplot grouped by first categorical/group field
        if 'group_field_id' in locals():
            plt.figure(figsize=(10,4))
            # Limit number of categories if too many
            nunique = filtered_df[group_field_id].nunique()
            if nunique > 10:
                common_cats = filtered_df[group_field_id].value_counts().index[:10]
                data = filtered_df[filtered_df[group_field_id].isin(common_cats)]
            else:
                data = filtered_df
            sns.boxplot(data=data, x=group_field_id, y=numeric_field_id)
            plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No suitable numeric field or filtered data for visualization.")
else:
    print("No data available to visualize.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a Croissant-defined open dataset using the `mlcroissant` library. By referencing all entities via their `@id` as prescribed by the schema, we programmatically:

- Explored dataset metadata and identified available entities;
- Loaded and previewed tabular data (via record sets or distributions);
- Performed exploratory analysis and visualized key variables.

This workflow can be adapted for any Croissant-compliant dataset, improving transparency, accessibility, and reproducible analytics for open data initiatives.
